Method: Historical Frequency Mapping. It ignores step-by-step movement. Instead, it simply calculates the single most frequently visited location for every specific time bin (e.g., "Mondays at 8:00 AM") in the past, and directly copies these static habits to predict future locations.

In [1]:
import pandas as pd
import numpy as np
import warnings

warnings.filterwarnings('ignore')

path = '/home/z5562390/work/dataset2023/dataset1.csv.gz'
output_path = 'Math(baseline).csv'

# --- Historical Pattern Extraction ---
def process_single_user(user_df):
    uid = user_df['uid'].iloc[0]
    
    # 1. Prepare training data (Days 0-59), keeping the last record per time bin
    train_df = user_df[user_df['d'] <= 59].drop_duplicates(subset=['d', 't'], keep='last')
    if train_df.empty: return pd.DataFrame()

    # 2. Build a full spatiotemporal grid and fill missing values
    full_index = pd.MultiIndex.from_product([[uid], range(60), range(48)], names=['uid', 'd', 't'])
    train_full = train_df.set_index(['uid', 'd', 't']).reindex(full_index)
    train_full = train_full.ffill().bfill().reset_index()
    train_full['weekday'] = train_full['d'] % 7
    
    # 3. Find the most frequent location for each weekday-time combination
    best_prob = train_full.groupby(['weekday', 't', 'x', 'y']).size().reset_index(name='count')\
                          .sort_values('count', ascending=False).drop_duplicates(['weekday', 't'])

    # 4. Generate future prediction grid (Days 60-74)
    pred_index = pd.MultiIndex.from_product([[uid], range(60, 75), range(48)], names=['uid', 'd', 't'])
    pred_df = pd.DataFrame(index=pred_index).reset_index()
    pred_df['weekday'] = pred_df['d'] % 7

    # 5. Map historical patterns to future predictions
    ans = pd.merge(pred_df, best_prob[['weekday', 't', 'x', 'y']], on=['weekday', 't'], how='left')
    
    return ans[['uid', 'd', 't', 'x', 'y']].astype(int)

# --- Streaming Execution Engine ---
# Initialize CSV with headers
pd.DataFrame(columns=['uid', 'd', 't', 'x', 'y']).to_csv(output_path, index=False)
dtypes = {'uid': np.int32, 'd': np.int16, 't': np.int8, 'x': np.int16, 'y': np.int16}
buffer_df = pd.DataFrame()

TARGET_USERS = 20000 
user_count, stop_reading = 0, False

print(f"Starting simplified calculation for {TARGET_USERS} users...")

# Process in chunks to manage memory
for chunk in pd.read_csv(path, usecols=['uid', 'd', 't', 'x', 'y'], dtype=dtypes, chunksize=500000):
    if stop_reading: break
        
    chunk = pd.concat([buffer_df, chunk], ignore_index=True)
    unique_uids = chunk['uid'].unique()
    
    # Buffer if chunk contains only one user to prevent data splitting
    if len(unique_uids) == 1:
        buffer_df = chunk
        continue
        
    # Hold back the last user to ensure completeness in the next chunk
    last_uid = unique_uids[-1]
    complete_data, buffer_df = chunk[chunk['uid'] != last_uid], chunk[chunk['uid'] == last_uid]
    
    for uid, user_data in complete_data.groupby('uid'):
        if user_count >= TARGET_USERS: 
            stop_reading = True
            break
            
        user_pred = process_single_user(user_data)
        if not user_pred.empty:
            # Save immediately to disk to free up RAM
            user_pred.to_csv(output_path, mode='a', header=False, index=False)
            
        user_count += 1
        if user_count % 1000 == 0: print(f"Processed {user_count} / {TARGET_USERS} users...")

# Process remaining buffered data
if not stop_reading and not buffer_df.empty and user_count < TARGET_USERS:
    for uid, user_data in buffer_df.groupby('uid'):
        user_pred = process_single_user(user_data)
        if not user_pred.empty: 
            user_pred.to_csv(output_path, mode='a', header=False, index=False)
        user_count += 1

print(f"\nTask Complete! Results saved to: {output_path}")

Starting simplified calculation for 20000 users...
Processed 1000 / 20000 users...
Processed 2000 / 20000 users...
Processed 3000 / 20000 users...
Processed 4000 / 20000 users...
Processed 5000 / 20000 users...
Processed 6000 / 20000 users...
Processed 7000 / 20000 users...
Processed 8000 / 20000 users...
Processed 9000 / 20000 users...
Processed 10000 / 20000 users...
Processed 11000 / 20000 users...
Processed 12000 / 20000 users...
Processed 13000 / 20000 users...
Processed 14000 / 20000 users...
Processed 15000 / 20000 users...
Processed 16000 / 20000 users...
Processed 17000 / 20000 users...
Processed 18000 / 20000 users...
Processed 19000 / 20000 users...
Processed 20000 / 20000 users...

Task Complete! Results saved to: Math(baseline).csv


In [2]:
import pandas as pd
import numpy as np
import warnings

warnings.filterwarnings('ignore')

# --- Configuration & File Paths ---
pred_path = 'Math(baseline).csv'
orig_path = '/home/z5562390/work/dataset2023/dataset1.csv.gz'
merged_output_path = 'predictions_vs_truth.csv'

# 1. Load predictions and identify target users
print("1. Loading predictions into memory...")
pred_df = pd.read_csv(pred_path)
target_uids = pred_df['uid'].unique()
print(f"-> Successfully loaded predictions for {len(target_uids)} users.")

# 2. Extract Ground Truth (Days 60-74) via chunking to manage memory
print("\n2. Extracting Ground Truth (Days 60-74) from massive dataset...")
gt_chunks = []
dtypes = {'uid': np.int32, 'd': np.int16, 't': np.int8, 'x': np.int16, 'y': np.int16}

for chunk in pd.read_csv(orig_path, usecols=['uid', 'd', 't', 'x', 'y'], dtype=dtypes, chunksize=1000000):
    # Filter by target dates and target users
    chunk = chunk[(chunk['d'] >= 60) & (chunk['d'] <= 74)]
    chunk = chunk[chunk['uid'].isin(target_uids)]
    
    # Keep the final settled location per time bin
    chunk = chunk.drop_duplicates(subset=['uid', 'd', 't'], keep='last')
    
    if not chunk.empty:
        gt_chunks.append(chunk)

gt_df = pd.concat(gt_chunks, ignore_index=True)
print(f"-> Extracted {len(gt_df)} real trajectory records for evaluation.")

# 3. Merge predictions with ground truth and calculate accuracy
print("\n3. Comparing Predictions vs. Ground Truth...")
eval_df = pd.merge(gt_df, pred_df, on=['uid', 'd', 't'], suffixes=('_true', '_pred'))

# Strict evaluation: Both X and Y must match exactly
eval_df['is_correct'] = (eval_df['x_true'] == eval_df['x_pred']) & (eval_df['y_true'] == eval_df['y_pred'])

accuracy = eval_df['is_correct'].mean() * 100
total_evaluated = len(eval_df)
correct_count = eval_df['is_correct'].sum()

print("==================================================")
print("📊 EVALUATION RESULTS")
print("==================================================")
print(f"Total Evaluated Records: {total_evaluated}")
print(f"Correct Predictions:     {correct_count}")
print(f"Overall Accuracy:        {accuracy:.2f} %")
print("==================================================")

# 4. Export comparison results for downstream error analysis
print(f"\n4. Saving merged results to {merged_output_path}...")
output_cols = ['uid', 'd', 't', 'x_true', 'y_true', 'x_pred', 'y_pred']
final_export_df = eval_df[output_cols]
final_export_df.to_csv(merged_output_path, index=False)

print("-> Save complete! You can now analyze the errors in the new CSV file.")

1. Loading predictions into memory...
-> Successfully loaded predictions for 20000 users.

2. Extracting Ground Truth (Days 60-74) from massive dataset...
-> Extracted 4861176 real trajectory records for evaluation.

3. Comparing Predictions vs. Ground Truth...
📊 EVALUATION RESULTS
Total Evaluated Records: 4861176
Correct Predictions:     1203463
Overall Accuracy:        24.76 %

4. Saving merged results to predictions_vs_truth.csv...
-> Save complete! You can now analyze the errors in the new CSV file.
